# Nextnano Simulation Results Browser

Explore one explicitly selected completed nextnano run without generating inputs or running a simulation.

## Setup

Import the public analysis APIs and locate the repository independently of the Jupyter launch directory.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.figure import Figure

from nextnanopp_tools import (
    apply_plot_font_scale,
    convergence_summary,
    find_probability_peaks,
    get_bias_dir,
    integrated_density_region_columns,
    list_variables,
    load_vtr_linecut,
    load_vtr_plane,
    plot_bias_volume_3d,
    plot_bias_volume_linecut,
    plot_bias_volume_slice,
    plot_convergence,
    plot_integrated_density_hole,
    plot_quantum_density_volume_3d,
    plot_quantum_density_volume_linecut,
    plot_quantum_density_volume_slice,
    plot_quantum_energy_spectrum,
    plot_quantum_occupation,
    plot_quantum_probability_volume_3d,
    plot_quantum_probability_volume_linecut,
    plot_quantum_probability_volume_slice,
    plot_total_charges,
    plot_vtr_slice,
    read_integrated_density_hole,
    read_quantum_energy_spectrum,
    read_quantum_occupation,
    read_total_charges,
    resolve_bias_output_file,
    resolve_quantum_output_file,
    resolve_quantum_probability_state_file,
    resolve_structure_file,
    validate_run_directory,
)

START_PATH = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (START_PATH, *START_PATH.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not locate the repository root from {START_PATH}; "
        "expected a parent containing pyproject.toml and src/."
    )

## User controls

Set one completed run and bias. The coordinates below are editable examples and should be adapted to the selected simulation.

In [ ]:
RUN_DIRECTORY = Path("path/to/completed/run")
BIAS = "bias_00000"

ANALYSE_QUANTUM_OUTPUTS = True
SHOW_OPTIONAL_3D = False
EXPORT_FIGURES = False
PLOT_FONT_SCALE = 1.15

# Editable example coordinates for the selected simulation.
XY_PLANE_Z_NM = -4.0
XZ_PLANE_Y_NM = 0.0

X_LINE_FIXED_COORDINATES_NM = {
    "y": 0.0,
    "z": -4.0,
}

Z_LINE_FIXED_COORDINATES_NM = {
    "x": 1150.0,
    "y": 0.0,
}

## Validate and inspect run

Validation requires the explicit run, selected bias, and canonical completion marker, without requiring every optional result up front.

In [ ]:
RUN_DIRECTORY = validate_run_directory(
    RUN_DIRECTORY,
    bias=BIAS,
    require_complete=True,
)
RUN_NAME = RUN_DIRECTORY.name
BIAS_DIRECTORY = get_bias_dir(RUN_DIRECTORY, BIAS)
COMPLETION_MARKER = RUN_DIRECTORY / "job_done.txt"

RUN_SUMMARY = pd.DataFrame(
    [
        {"item": "normalized run directory", "value": str(RUN_DIRECTORY)},
        {"item": "run name", "value": RUN_NAME},
        {"item": "selected bias", "value": BIAS},
        {"item": "bias directory", "value": str(BIAS_DIRECTORY)},
        {"item": "completion marker present", "value": COMPLETION_MARKER.is_file()},
    ]
)
display(RUN_SUMMARY)

In [ ]:
CLASSICAL_OUTPUT_QUANTITIES = {
    "potential": "potential.vtr",
    "bandedges": "bandedges.vtr",
    "density_hole": "density_hole.vtr",
}
CLASSICAL_OUTPUT_FILES = {}
classical_output_records = []

for quantity, expected_filename in CLASSICAL_OUTPUT_QUANTITIES.items():
    try:
        resolved_file = resolve_bias_output_file(
            RUN_DIRECTORY,
            quantity,
            bias=BIAS,
            preferred_extensions=("vtr",),
        )
    except FileNotFoundError as output_error:
        classical_output_records.append(
            {
                "quantity": quantity,
                "expected file": expected_filename,
                "status": "absent",
                "file": "",
                "variables": "",
            }
        )
        display(f"Optional classical category {quantity!r} is unavailable: {output_error}")
    else:
        CLASSICAL_OUTPUT_FILES[quantity] = resolved_file
        classical_output_records.append(
            {
                "quantity": quantity,
                "expected file": expected_filename,
                "status": "available",
                "file": str(resolved_file.relative_to(BIAS_DIRECTORY)),
                "variables": ", ".join(list_variables(resolved_file)),
            }
        )

CLASSICAL_OUTPUT_INVENTORY = pd.DataFrame(classical_output_records)
display(CLASSICAL_OUTPUT_INVENTORY)

## Structure and diagnostics

Each optional diagnostic reports a missing file locally so other sections remain usable.

In [ ]:
STRUCTURE_XY_FIGURE = None
try:
    QUANTITY = "regions_all"
    RESOLVED_FILE = resolve_structure_file(
        RUN_DIRECTORY,
        QUANTITY,
        preferred_extensions=("vtr",),
    )
    VARIABLE = list_variables(RESOLVED_FILE)[0]
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM
    LOG10 = False

    STRUCTURE_XY_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    STRUCTURE_XY_FIGURE = plot_vtr_slice(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            "Simulated structure at "
            f"z = {STRUCTURE_XY_DATA['slice_coordinate']:g} nm"
        ),
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(STRUCTURE_XY_FIGURE)
except (FileNotFoundError, ValueError) as diagnostic_error:
    display(f"Structure xy view unavailable: {diagnostic_error}")

In [ ]:
STRUCTURE_XZ_FIGURE = None
try:
    QUANTITY = "materials"
    RESOLVED_FILE = resolve_structure_file(
        RUN_DIRECTORY,
        QUANTITY,
        preferred_extensions=("vtr",),
    )
    VARIABLE = list_variables(RESOLVED_FILE)[0]
    SLICE_AXIS = "y"
    SLICE_COORDINATE_NM = XZ_PLANE_Y_NM
    LOG10 = False

    STRUCTURE_XZ_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    STRUCTURE_XZ_FIGURE = plot_vtr_slice(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            "Simulated structure at "
            f"y = {STRUCTURE_XZ_DATA['slice_coordinate']:g} nm"
        ),
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(STRUCTURE_XZ_FIGURE)
except (FileNotFoundError, ValueError) as diagnostic_error:
    display(f"Structure xz view unavailable: {diagnostic_error}")

In [ ]:
CONVERGENCE_FIGURE = None
try:
    CONVERGENCE_SUMMARY = convergence_summary(BIAS_DIRECTORY)
    CONVERGENCE_FIGURE = plot_convergence(
        BIAS_DIRECTORY,
        interactive=False,
        font_scale=PLOT_FONT_SCALE,
    )
    display(CONVERGENCE_SUMMARY)
    display(CONVERGENCE_FIGURE)
except (FileNotFoundError, ValueError) as diagnostic_error:
    display(f"Convergence diagnostic unavailable: {diagnostic_error}")

In [ ]:
INTEGRATED_HOLE_DENSITY_FIGURE = None
try:
    INTEGRATED_HOLE_DENSITY = read_integrated_density_hole(RUN_DIRECTORY)
    INTEGRATED_REGION_COLUMNS = integrated_density_region_columns(
        INTEGRATED_HOLE_DENSITY
    )
    display(INTEGRATED_HOLE_DENSITY.loc[:, INTEGRATED_REGION_COLUMNS].tail(1))
    INTEGRATED_HOLE_DENSITY_FIGURE = plot_integrated_density_hole(
        RUN_DIRECTORY,
        region_columns=INTEGRATED_REGION_COLUMNS,
        interactive=False,
        label_with_materials=False,
        font_scale=PLOT_FONT_SCALE,
    )
    display(INTEGRATED_HOLE_DENSITY_FIGURE)
except (FileNotFoundError, ValueError, IndexError) as diagnostic_error:
    display(f"Integrated-hole-density diagnostic unavailable: {diagnostic_error}")

In [ ]:
TOTAL_CHARGES_FIGURE = None
try:
    TOTAL_CHARGES = read_total_charges(BIAS_DIRECTORY)
    TOTAL_CHARGES_FIGURE = plot_total_charges(
        BIAS_DIRECTORY,
        interactive=False,
        font_scale=PLOT_FONT_SCALE,
    )
    display(TOTAL_CHARGES)
    display(TOTAL_CHARGES_FIGURE)
except (FileNotFoundError, ValueError, KeyError) as diagnostic_error:
    display(f"Total-charge diagnostic unavailable: {diagnostic_error}")

## Classical outputs

Each cell resolves its own required result and exposes the quantity, variable, cut, and display settings for editing.

### Hole density

The examples use linear scale and report the nearest grid coordinate actually selected.

In [ ]:
QUANTITY = "density_hole"
VARIABLE = "Hole_density"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM
INTERACTIVE = False
LOG10 = False

HOLE_DENSITY_XY_DATA = load_vtr_plane(
    RESOLVED_FILE,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
HOLE_DENSITY_XY_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Classical hole density at "
        f"z = {HOLE_DENSITY_XY_DATA['slice_coordinate']:g} nm"
    ),
    interactive=INTERACTIVE,
    log10=LOG10,
    font_scale=PLOT_FONT_SCALE,
)
display(HOLE_DENSITY_XY_FIGURE)

In [ ]:
QUANTITY = "density_hole"
VARIABLE = "Hole_density"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
LINE_AXIS = "x"
FIXED_COORDINATES_NM = dict(X_LINE_FIXED_COORDINATES_NM)
INTERACTIVE = False
YSCALE = "linear"

HOLE_DENSITY_X_LINE_DATA = load_vtr_linecut(
    RESOLVED_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)
HOLE_DENSITY_X_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=INTERACTIVE,
    yscale=YSCALE,
    font_scale=PLOT_FONT_SCALE,
)
HOLE_DENSITY_X_LINE_FIGURE.axes[0].title.set_text(
    f"Classical hole density along {LINE_AXIS} at "
    f"y = {HOLE_DENSITY_X_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {HOLE_DENSITY_X_LINE_DATA['chosen_coords']['z']:g} nm"
)
display(HOLE_DENSITY_X_LINE_FIGURE)

In [ ]:
QUANTITY = "density_hole"
VARIABLE = "Hole_density"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "y"
SLICE_COORDINATE_NM = XZ_PLANE_Y_NM
INTERACTIVE = False
LOG10 = False

HOLE_DENSITY_XZ_DATA = load_vtr_plane(
    RESOLVED_FILE,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
HOLE_DENSITY_XZ_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Classical hole density at "
        f"y = {HOLE_DENSITY_XZ_DATA['slice_coordinate']:g} nm"
    ),
    interactive=INTERACTIVE,
    log10=LOG10,
    font_scale=PLOT_FONT_SCALE,
)
display(HOLE_DENSITY_XZ_FIGURE)

### Electrostatic potential

Plane and line examples use the same editable coordinates from the user-control cell.

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM
INTERACTIVE = False
LOG10 = False

POTENTIAL_XY_DATA = load_vtr_plane(
    RESOLVED_FILE,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
POTENTIAL_XY_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Electrostatic potential at "
        f"z = {POTENTIAL_XY_DATA['slice_coordinate']:g} nm"
    ),
    interactive=INTERACTIVE,
    log10=LOG10,
    font_scale=PLOT_FONT_SCALE,
)
display(POTENTIAL_XY_FIGURE)

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "y"
SLICE_COORDINATE_NM = XZ_PLANE_Y_NM
INTERACTIVE = False
LOG10 = False

POTENTIAL_XZ_DATA = load_vtr_plane(
    RESOLVED_FILE,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
POTENTIAL_XZ_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Electrostatic potential at "
        f"y = {POTENTIAL_XZ_DATA['slice_coordinate']:g} nm"
    ),
    interactive=INTERACTIVE,
    log10=LOG10,
    font_scale=PLOT_FONT_SCALE,
)
display(POTENTIAL_XZ_FIGURE)

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
LINE_AXIS = "x"
FIXED_COORDINATES_NM = dict(X_LINE_FIXED_COORDINATES_NM)
INTERACTIVE = False
YSCALE = "linear"

POTENTIAL_X_LINE_DATA = load_vtr_linecut(
    RESOLVED_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)
POTENTIAL_X_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=INTERACTIVE,
    yscale=YSCALE,
    font_scale=PLOT_FONT_SCALE,
)
POTENTIAL_X_LINE_FIGURE.axes[0].title.set_text(
    f"Electrostatic potential along {LINE_AXIS} at "
    f"y = {POTENTIAL_X_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {POTENTIAL_X_LINE_DATA['chosen_coords']['z']:g} nm"
)
display(POTENTIAL_X_LINE_FIGURE)

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
LINE_AXIS = "z"
FIXED_COORDINATES_NM = dict(Z_LINE_FIXED_COORDINATES_NM)
INTERACTIVE = False
YSCALE = "linear"

POTENTIAL_Z_LINE_DATA = load_vtr_linecut(
    RESOLVED_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)
POTENTIAL_Z_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=INTERACTIVE,
    yscale=YSCALE,
    font_scale=PLOT_FONT_SCALE,
)
POTENTIAL_Z_LINE_FIGURE.axes[0].title.set_text(
    f"Electrostatic potential along {LINE_AXIS} at "
    f"x = {POTENTIAL_Z_LINE_DATA['chosen_coords']['x']:g} nm, "
    f"y = {POTENTIAL_Z_LINE_DATA['chosen_coords']['y']:g} nm"
)
display(POTENTIAL_Z_LINE_FIGURE)

### Band edges

The primary examples use `HH`; the final cell is an editable example for another variable reported by the selected `bandedges.vtr`.

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM
INTERACTIVE = False
LOG10 = False

HH_BAND_XY_DATA = load_vtr_plane(
    RESOLVED_FILE,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
HH_BAND_XY_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=f"HH band edge at z = {HH_BAND_XY_DATA['slice_coordinate']:g} nm",
    interactive=INTERACTIVE,
    log10=LOG10,
    font_scale=PLOT_FONT_SCALE,
)
display(HH_BAND_XY_FIGURE)

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "y"
SLICE_COORDINATE_NM = XZ_PLANE_Y_NM
INTERACTIVE = False
LOG10 = False

HH_BAND_XZ_DATA = load_vtr_plane(
    RESOLVED_FILE,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
HH_BAND_XZ_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=f"HH band edge at y = {HH_BAND_XZ_DATA['slice_coordinate']:g} nm",
    interactive=INTERACTIVE,
    log10=LOG10,
    font_scale=PLOT_FONT_SCALE,
)
display(HH_BAND_XZ_FIGURE)

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
LINE_AXIS = "x"
FIXED_COORDINATES_NM = dict(X_LINE_FIXED_COORDINATES_NM)
INTERACTIVE = False
YSCALE = "linear"

HH_BAND_X_LINE_DATA = load_vtr_linecut(
    RESOLVED_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)
HH_BAND_X_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=INTERACTIVE,
    yscale=YSCALE,
    font_scale=PLOT_FONT_SCALE,
)
HH_BAND_X_LINE_FIGURE.axes[0].title.set_text(
    f"HH band edge along {LINE_AXIS} at "
    f"y = {HH_BAND_X_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {HH_BAND_X_LINE_DATA['chosen_coords']['z']:g} nm"
)
display(HH_BAND_X_LINE_FIGURE)

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
LINE_AXIS = "z"
FIXED_COORDINATES_NM = dict(Z_LINE_FIXED_COORDINATES_NM)
INTERACTIVE = False
YSCALE = "linear"

HH_BAND_Z_LINE_DATA = load_vtr_linecut(
    RESOLVED_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)
HH_BAND_Z_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=INTERACTIVE,
    yscale=YSCALE,
    font_scale=PLOT_FONT_SCALE,
)
HH_BAND_Z_LINE_FIGURE.axes[0].title.set_text(
    f"HH band edge along {LINE_AXIS} at "
    f"x = {HH_BAND_Z_LINE_DATA['chosen_coords']['x']:g} nm, "
    f"y = {HH_BAND_Z_LINE_DATA['chosen_coords']['y']:g} nm"
)
display(HH_BAND_Z_LINE_FIGURE)

In [ ]:
# Replace this example with any variable listed for bandedges.vtr.
QUANTITY = "bandedges"
VARIABLE = "LH"
RESOLVED_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM
INTERACTIVE = False
LOG10 = False
OTHER_BAND_XY_FIGURE = None

if VARIABLE in list_variables(RESOLVED_FILE):
    OTHER_BAND_XY_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    OTHER_BAND_XY_FIGURE = plot_bias_volume_slice(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"{VARIABLE} band edge at "
            f"z = {OTHER_BAND_XY_DATA['slice_coordinate']:g} nm"
        ),
        interactive=INTERACTIVE,
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(OTHER_BAND_XY_FIGURE)
else:
    display(
        f"Band-edge variable {VARIABLE!r} is unavailable; "
        f"choose from {list_variables(RESOLVED_FILE)}."
    )

## Quantum outputs

Quantum paths and data are resolved only when quantum analysis is enabled.

In [ ]:
QUANTUM_REGION = "c-Ge_QW"
QUANTUM_BAND = "HH"
QUANTUM_KPOINT = "k00000"
QUANTUM_STATE = 1

if ANALYSE_QUANTUM_OUTPUTS:
    QUANTUM_EXPECTED_DIRECTORY = (
        BIAS_DIRECTORY / "Quantum" / QUANTUM_REGION / QUANTUM_BAND
    )
    quantum_output_records = [
        {
            "result": "expected region/band directory",
            "status": (
                "available" if QUANTUM_EXPECTED_DIRECTORY.is_dir() else "absent"
            ),
            "file": str(QUANTUM_EXPECTED_DIRECTORY.relative_to(BIAS_DIRECTORY)),
            "variables": "",
        }
    ]

    try:
        quantum_density_file = resolve_quantum_output_file(
            RUN_DIRECTORY,
            "density",
            region=QUANTUM_REGION,
            band=QUANTUM_BAND,
            bias=BIAS,
            preferred_extensions=("vtr",),
        )
    except FileNotFoundError as output_error:
        quantum_output_records.append(
            {"result": "quantum density", "status": "absent", "file": "", "variables": ""}
        )
        display(f"Quantum density is unavailable: {output_error}")
    else:
        quantum_output_records.append(
            {
                "result": "quantum density",
                "status": "available",
                "file": str(quantum_density_file.relative_to(BIAS_DIRECTORY)),
                "variables": ", ".join(list_variables(quantum_density_file)),
            }
        )

    try:
        quantum_probability_file = resolve_quantum_probability_state_file(
            RUN_DIRECTORY,
            state=QUANTUM_STATE,
            region=QUANTUM_REGION,
            band=QUANTUM_BAND,
            kpoint=QUANTUM_KPOINT,
            shifted=True,
            bias=BIAS,
            preferred_extensions=("vtr",),
        )
    except FileNotFoundError as output_error:
        quantum_output_records.append(
            {"result": "shifted probability", "status": "absent", "file": "", "variables": ""}
        )
        display(f"Shifted probability output is unavailable: {output_error}")
    else:
        quantum_output_records.append(
            {
                "result": "shifted probability",
                "status": "available",
                "file": str(quantum_probability_file.relative_to(BIAS_DIRECTORY)),
                "variables": ", ".join(list_variables(quantum_probability_file)),
            }
        )

    QUANTUM_OUTPUT_INVENTORY = pd.DataFrame(quantum_output_records)
    display(QUANTUM_OUTPUT_INVENTORY)
else:
    display("Quantum analysis is disabled (ANALYSE_QUANTUM_OUTPUTS=False).")

### Quantum-calculated hole density

The configured region and band are shown in xy, x-line, and xz examples.

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "density"
    VARIABLE = "Density"
    RESOLVED_FILE = resolve_quantum_output_file(
        RUN_DIRECTORY,
        QUANTITY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM
    INTERACTIVE = False
    LOG10 = False

    QUANTUM_DENSITY_XY_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    QUANTUM_DENSITY_XY_FIGURE = plot_quantum_density_volume_slice(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"Quantum-calculated {QUANTUM_BAND} hole density at "
            f"z = {QUANTUM_DENSITY_XY_DATA['slice_coordinate']:g} nm"
        ),
        interactive=INTERACTIVE,
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_DENSITY_XY_FIGURE)

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "density"
    VARIABLE = "Density"
    RESOLVED_FILE = resolve_quantum_output_file(
        RUN_DIRECTORY,
        QUANTITY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    LINE_AXIS = "x"
    FIXED_COORDINATES_NM = dict(X_LINE_FIXED_COORDINATES_NM)
    INTERACTIVE = False
    YSCALE = "linear"

    QUANTUM_DENSITY_X_LINE_DATA = load_vtr_linecut(
        RESOLVED_FILE,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
    )
    QUANTUM_DENSITY_X_LINE_FIGURE = plot_quantum_density_volume_linecut(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
        title=(
            f"Quantum-calculated {QUANTUM_BAND} hole density along {LINE_AXIS} at "
            f"y = {QUANTUM_DENSITY_X_LINE_DATA['chosen_coords']['y']:g} nm, "
            f"z = {QUANTUM_DENSITY_X_LINE_DATA['chosen_coords']['z']:g} nm"
        ),
        interactive=INTERACTIVE,
        yscale=YSCALE,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_DENSITY_X_LINE_FIGURE)

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "density"
    VARIABLE = "Density"
    RESOLVED_FILE = resolve_quantum_output_file(
        RUN_DIRECTORY,
        QUANTITY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    SLICE_AXIS = "y"
    SLICE_COORDINATE_NM = XZ_PLANE_Y_NM
    INTERACTIVE = False
    LOG10 = False

    QUANTUM_DENSITY_XZ_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    QUANTUM_DENSITY_XZ_FIGURE = plot_quantum_density_volume_slice(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"Quantum-calculated {QUANTUM_BAND} hole density at "
            f"y = {QUANTUM_DENSITY_XZ_DATA['slice_coordinate']:g} nm"
        ),
        interactive=INTERACTIVE,
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_DENSITY_XZ_FIGURE)

### Wavefunction probability

These cells inspect the configured shifted state; separated maxima are probability-density peaks, not definitive dot centres.

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    RESOLVED_FILE = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM
    INTERACTIVE = False
    LOG10 = False

    QUANTUM_PROBABILITY_XY_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    QUANTUM_PROBABILITY_XY_FIGURE = plot_quantum_probability_volume_slice(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"Shifted probability for state {QUANTUM_STATE} at "
            f"z = {QUANTUM_PROBABILITY_XY_DATA['slice_coordinate']:g} nm"
        ),
        interactive=INTERACTIVE,
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_PROBABILITY_XY_FIGURE)

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    RESOLVED_FILE = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    LINE_AXIS = "x"
    FIXED_COORDINATES_NM = dict(X_LINE_FIXED_COORDINATES_NM)
    INTERACTIVE = False
    YSCALE = "linear"

    QUANTUM_PROBABILITY_X_LINE_DATA = load_vtr_linecut(
        RESOLVED_FILE,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
    )
    QUANTUM_PROBABILITY_X_LINE_FIGURE = plot_quantum_probability_volume_linecut(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
        title=(
            f"Shifted probability for state {QUANTUM_STATE} along {LINE_AXIS} at "
            f"y = {QUANTUM_PROBABILITY_X_LINE_DATA['chosen_coords']['y']:g} nm, "
            f"z = {QUANTUM_PROBABILITY_X_LINE_DATA['chosen_coords']['z']:g} nm"
        ),
        interactive=INTERACTIVE,
        yscale=YSCALE,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_PROBABILITY_X_LINE_FIGURE)

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    RESOLVED_FILE = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    SLICE_AXIS = "y"
    SLICE_COORDINATE_NM = XZ_PLANE_Y_NM
    INTERACTIVE = False
    LOG10 = False

    QUANTUM_PROBABILITY_XZ_DATA = load_vtr_plane(
        RESOLVED_FILE,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    QUANTUM_PROBABILITY_XZ_FIGURE = plot_quantum_probability_volume_slice(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"Shifted probability for state {QUANTUM_STATE} at "
            f"y = {QUANTUM_PROBABILITY_XZ_DATA['slice_coordinate']:g} nm"
        ),
        interactive=INTERACTIVE,
        log10=LOG10,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_PROBABILITY_XZ_FIGURE)

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    RESOLVED_FILE = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    N_PROBABILITY_PEAKS = 2
    MIN_LATERAL_PEAK_SEPARATION_NM = 50.0

    QUANTUM_PROBABILITY_PEAKS = find_probability_peaks(
        RESOLVED_FILE,
        variable=VARIABLE,
        n_peaks=N_PROBABILITY_PEAKS,
        min_lateral_separation_nm=MIN_LATERAL_PEAK_SEPARATION_NM,
    )
    QUANTUM_PROBABILITY_PEAK_SUMMARY = QUANTUM_PROBABILITY_PEAKS.loc[
        :, ["peak", "x_nm", "y_nm", "z_nm", "probability"]
    ]
    display("Separated probability-density maxima; these are not definitive dot centres.")
    display(QUANTUM_PROBABILITY_PEAK_SUMMARY)

### Occupation and energy spectrum

Tables are kept concise and paired with static figures.

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTUM_OCCUPATION = read_quantum_occupation(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
    )
    QUANTUM_OCCUPATION_COLUMNS = list(
        dict.fromkeys(
            (QUANTUM_OCCUPATION.columns[0], QUANTUM_OCCUPATION.columns[-1])
        )
    )
    display(QUANTUM_OCCUPATION.loc[:, QUANTUM_OCCUPATION_COLUMNS].head(10))
    QUANTUM_OCCUPATION_FIGURE = plot_quantum_occupation(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        interactive=False,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_OCCUPATION_FIGURE)

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTUM_ENERGY_SPECTRUM = read_quantum_energy_spectrum(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
    )
    QUANTUM_ENERGY_COLUMNS = list(
        dict.fromkeys(
            (QUANTUM_ENERGY_SPECTRUM.columns[0], QUANTUM_ENERGY_SPECTRUM.columns[-1])
        )
    )
    display(QUANTUM_ENERGY_SPECTRUM.loc[:, QUANTUM_ENERGY_COLUMNS].head(10))
    QUANTUM_ENERGY_FIGURE = plot_quantum_energy_spectrum(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        interactive=False,
        font_scale=PLOT_FONT_SCALE,
    )
    display(QUANTUM_ENERGY_FIGURE)

## Multi-quantity line comparison

The panels use one requested x cut. Each quantity remains on its native x-grid without interpolation or resampling, while all panels display the common overlapping x-range. Energy, density, and probability remain on separate labelled axes; raw values are not normalized or mixed across incompatible units.

In [ ]:
LINE_AXIS = "x"
FIXED_COORDINATES_NM = dict(X_LINE_FIXED_COORDINATES_NM)

QUANTITY = "bandedges"
VARIABLE = "HH"
BANDEDGES_COMPARISON_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
HH_COMPARISON_LINE_DATA = load_vtr_linecut(
    BANDEDGES_COMPARISON_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)

BANDEDGE_VARIABLES = list_variables(BANDEDGES_COMPARISON_FILE)
FERMI_VARIABLE = (
    "hole_Fermi_level" if "hole_Fermi_level" in BANDEDGE_VARIABLES else None
)
FERMI_COMPARISON_LINE_DATA = None
if FERMI_VARIABLE is not None:
    fermi_candidate = load_vtr_linecut(
        BANDEDGES_COMPARISON_FILE,
        variable=FERMI_VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
    )
    if fermi_candidate["variable"].unit == HH_COMPARISON_LINE_DATA["variable"].unit:
        FERMI_COMPARISON_LINE_DATA = fermi_candidate
    else:
        display("Hole Fermi level omitted because its unit does not match the HH energy unit.")
else:
    display("hole_Fermi_level is not present in the selected bandedges output; curve omitted.")

QUANTITY = "density_hole"
VARIABLE = "Hole_density"
CLASSICAL_DENSITY_COMPARISON_FILE = resolve_bias_output_file(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    preferred_extensions=("vtr",),
)
CLASSICAL_DENSITY_COMPARISON_LINE_DATA = load_vtr_linecut(
    CLASSICAL_DENSITY_COMPARISON_FILE,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)

QUANTUM_DENSITY_COMPARISON_LINE_DATA = None
PROBABILITY_COMPARISON_LINE_DATA = None
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "density"
    VARIABLE = "Density"
    QUANTUM_DENSITY_COMPARISON_FILE = resolve_quantum_output_file(
        RUN_DIRECTORY,
        QUANTITY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    QUANTUM_DENSITY_COMPARISON_LINE_DATA = load_vtr_linecut(
        QUANTUM_DENSITY_COMPARISON_FILE,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
    )

    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    PROBABILITY_COMPARISON_FILE = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    PROBABILITY_COMPARISON_LINE_DATA = load_vtr_linecut(
        PROBABILITY_COMPARISON_FILE,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
    )

comparison_lines = {
    "HH band edge": HH_COMPARISON_LINE_DATA,
    "classical hole density": CLASSICAL_DENSITY_COMPARISON_LINE_DATA,
}
if FERMI_COMPARISON_LINE_DATA is not None:
    comparison_lines["hole Fermi level"] = FERMI_COMPARISON_LINE_DATA
if QUANTUM_DENSITY_COMPARISON_LINE_DATA is not None:
    comparison_lines["quantum hole density"] = QUANTUM_DENSITY_COMPARISON_LINE_DATA
if PROBABILITY_COMPARISON_LINE_DATA is not None:
    comparison_lines["wavefunction probability"] = PROBABILITY_COMPARISON_LINE_DATA

for line_label, line_data in comparison_lines.items():
    for coordinate_name in ("y", "z"):
        reference_coordinate = HH_COMPARISON_LINE_DATA["chosen_coords"][coordinate_name]
        if not np.isclose(line_data["chosen_coords"][coordinate_name], reference_coordinate):
            raise ValueError(
                f"{line_label} selected a different {coordinate_name} coordinate; "
                "adjust the requested cut to a shared grid point."
            )

X_OVERLAP_MIN_NM = max(
    float(np.min(line_data["axis"]))
    for line_data in comparison_lines.values()
)
X_OVERLAP_MAX_NM = min(
    float(np.max(line_data["axis"]))
    for line_data in comparison_lines.values()
)
if X_OVERLAP_MIN_NM >= X_OVERLAP_MAX_NM:
    raise ValueError(
        "The selected quantities have no common overlapping x-range on their native grids."
    )

COMPARISON_PANEL_COUNT = 2 + (2 if ANALYSE_QUANTUM_OUTPUTS else 0)
multi_quantity_x_line_figure, comparison_axes = plt.subplots(
    COMPARISON_PANEL_COUNT,
    1,
    figsize=(11, 2.8 * COMPARISON_PANEL_COUNT),
    sharex=True,
)
comparison_axes = np.atleast_1d(comparison_axes)

energy_axis = comparison_axes[0]
energy_axis.plot(
    HH_COMPARISON_LINE_DATA["axis"],
    HH_COMPARISON_LINE_DATA["values"],
    label="HH band edge",
)
if FERMI_COMPARISON_LINE_DATA is not None:
    energy_axis.plot(
        FERMI_COMPARISON_LINE_DATA["axis"],
        FERMI_COMPARISON_LINE_DATA["values"],
        linestyle="--",
        label="hole Fermi level",
    )
energy_axis.set_ylabel(HH_COMPARISON_LINE_DATA["variable"].label or "Energy")
energy_axis.legend()
energy_axis.grid(alpha=0.25)

classical_density_axis = comparison_axes[1]
classical_density_axis.plot(
    CLASSICAL_DENSITY_COMPARISON_LINE_DATA["axis"],
    CLASSICAL_DENSITY_COMPARISON_LINE_DATA["values"],
    color="tab:blue",
    label="classical hole density",
)
classical_density_axis.set_ylabel(
    CLASSICAL_DENSITY_COMPARISON_LINE_DATA["variable"].label or "Classical density"
)
classical_density_axis.legend()
classical_density_axis.grid(alpha=0.25)

if ANALYSE_QUANTUM_OUTPUTS:
    quantum_density_axis = comparison_axes[2]
    quantum_density_axis.plot(
        QUANTUM_DENSITY_COMPARISON_LINE_DATA["axis"],
        QUANTUM_DENSITY_COMPARISON_LINE_DATA["values"],
        color="tab:orange",
        label=f"quantum {QUANTUM_BAND} hole density",
    )
    quantum_density_axis.set_ylabel(
        QUANTUM_DENSITY_COMPARISON_LINE_DATA["variable"].label or "Quantum density"
    )
    quantum_density_axis.legend()
    quantum_density_axis.grid(alpha=0.25)

    probability_axis = comparison_axes[3]
    probability_axis.plot(
        PROBABILITY_COMPARISON_LINE_DATA["axis"],
        PROBABILITY_COMPARISON_LINE_DATA["values"],
        color="tab:green",
        label=f"state {QUANTUM_STATE} probability",
    )
    probability_axis.set_ylabel(
        PROBABILITY_COMPARISON_LINE_DATA["variable"].label or "Probability"
    )
    probability_axis.legend()
    probability_axis.grid(alpha=0.25)

for comparison_axis in comparison_axes:
    comparison_axis.set_xlim(X_OVERLAP_MIN_NM, X_OVERLAP_MAX_NM)

comparison_axes[-1].set_xlabel(HH_COMPARISON_LINE_DATA["axis_label"])
multi_quantity_x_line_figure.suptitle(
    "Raw quantities on native grids over the common range "
    f"x = [{X_OVERLAP_MIN_NM:g}, {X_OVERLAP_MAX_NM:g}] nm at "
    f"y = {HH_COMPARISON_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {HH_COMPARISON_LINE_DATA['chosen_coords']['z']:g} nm"
)
apply_plot_font_scale(
    multi_quantity_x_line_figure,
    font_scale=PLOT_FONT_SCALE,
)
multi_quantity_x_line_figure.tight_layout()
display(multi_quantity_x_line_figure)

## Optional 3D exploration

These Plotly views are disabled by default and are intentionally excluded from static export.

In [ ]:
if SHOW_OPTIONAL_3D:
    QUANTITY = "density_hole"
    VARIABLE = "Hole_density"
    COORDINATE_RANGES_NM = {"z": (-30.0, 10.0)}
    LOG10 = False
    VOLUME_MODE = "volume"

    CLASSICAL_HOLE_DENSITY_3D_FIGURE = plot_bias_volume_3d(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        variable=VARIABLE,
        title="Classical hole density near the selected quantum-well depth",
        log10=LOG10,
        coord_ranges=COORDINATE_RANGES_NM,
        max_points=150_000,
        mode=VOLUME_MODE,
        opacity=0.18,
        surface_count=10,
        font_scale=PLOT_FONT_SCALE,
    )

In [ ]:
if SHOW_OPTIONAL_3D:
    QUANTITY = "potential"
    VARIABLE = "Potential"
    COORDINATE_RANGES_NM = {"z": (-30.0, 10.0)}
    LOG10 = False
    VOLUME_MODE = "volume"

    POTENTIAL_3D_FIGURE = plot_bias_volume_3d(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        variable=VARIABLE,
        title="Electrostatic potential near the selected quantum-well depth",
        log10=LOG10,
        coord_ranges=COORDINATE_RANGES_NM,
        max_points=150_000,
        mode=VOLUME_MODE,
        opacity=0.18,
        surface_count=10,
        font_scale=PLOT_FONT_SCALE,
    )

In [ ]:
if SHOW_OPTIONAL_3D:
    QUANTITY = "bandedges"
    VARIABLE = "HH"
    COORDINATE_RANGES_NM = {"z": (-30.0, 10.0)}
    LOG10 = False
    VOLUME_MODE = "volume"

    HH_BAND_EDGE_3D_FIGURE = plot_bias_volume_3d(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        variable=VARIABLE,
        title="HH band edge near the selected quantum-well depth",
        log10=LOG10,
        coord_ranges=COORDINATE_RANGES_NM,
        max_points=150_000,
        mode=VOLUME_MODE,
        opacity=0.18,
        surface_count=10,
        font_scale=PLOT_FONT_SCALE,
    )

In [ ]:
if SHOW_OPTIONAL_3D:
    if ANALYSE_QUANTUM_OUTPUTS:
        QUANTITY = "density"
        VARIABLE = "Density"
        COORDINATE_RANGES_NM = {"z": (-30.0, 10.0)}
        LOG10 = False
        VOLUME_MODE = "volume"

        QUANTUM_DENSITY_3D_FIGURE = plot_quantum_density_volume_3d(
            RUN_DIRECTORY,
            region=QUANTUM_REGION,
            band=QUANTUM_BAND,
            bias=BIAS,
            variable=VARIABLE,
            title=f"Quantum-calculated {QUANTUM_BAND} hole density in 3D",
            log10=LOG10,
            coord_ranges=COORDINATE_RANGES_NM,
            max_points=150_000,
            mode=VOLUME_MODE,
            opacity=0.18,
            surface_count=10,
            font_scale=PLOT_FONT_SCALE,
        )

In [ ]:
if SHOW_OPTIONAL_3D:
    if ANALYSE_QUANTUM_OUTPUTS:
        QUANTITY = "probability_shift"
        VARIABLE = f"Psi^2_{QUANTUM_STATE}"
        COORDINATE_RANGES_NM = {"z": (-30.0, 10.0)}
        LOG10 = False
        VOLUME_MODE = "volume"

        QUANTUM_PROBABILITY_3D_FIGURE = plot_quantum_probability_volume_3d(
            RUN_DIRECTORY,
            state=QUANTUM_STATE,
            region=QUANTUM_REGION,
            band=QUANTUM_BAND,
            kpoint=QUANTUM_KPOINT,
            shifted=True,
            bias=BIAS,
            variable=VARIABLE,
            title=f"Shifted probability for state {QUANTUM_STATE} in 3D",
            log10=LOG10,
            coord_ranges=COORDINATE_RANGES_NM,
            max_points=150_000,
            mode=VOLUME_MODE,
            opacity=0.18,
            surface_count=10,
            font_scale=PLOT_FONT_SCALE,
        )

## Selected figure export

Edit `SELECTED_FIGURES` before enabling export. Optional diagnostics are added only when their Matplotlib figures exist; Plotly views are never selected.

In [ ]:
# Comment out or delete entries that should not be exported.
SELECTED_FIGURES = {
    "classical_hole_density_xy": HOLE_DENSITY_XY_FIGURE,
    "classical_hole_density_x_line": HOLE_DENSITY_X_LINE_FIGURE,
    "electrostatic_potential_xy": POTENTIAL_XY_FIGURE,
    "electrostatic_potential_xz": POTENTIAL_XZ_FIGURE,
    "electrostatic_potential_x_line": POTENTIAL_X_LINE_FIGURE,
    "hh_band_edge_xy": HH_BAND_XY_FIGURE,
    "hh_band_edge_xz": HH_BAND_XZ_FIGURE,
    "hh_band_edge_x_line": HH_BAND_X_LINE_FIGURE,
    "hh_band_edge_z_line": HH_BAND_Z_LINE_FIGURE,
    "multi_quantity_x_line": multi_quantity_x_line_figure,
}

for figure_stem, optional_figure in (
    ("structure_xy", STRUCTURE_XY_FIGURE),
    ("structure_xz", STRUCTURE_XZ_FIGURE),
    ("convergence", CONVERGENCE_FIGURE),
    ("integrated_hole_density", INTEGRATED_HOLE_DENSITY_FIGURE),
    ("total_charge", TOTAL_CHARGES_FIGURE),
):
    if isinstance(optional_figure, Figure):
        SELECTED_FIGURES[figure_stem] = optional_figure

if ANALYSE_QUANTUM_OUTPUTS:
    SELECTED_FIGURES.update(
        {
            f"quantum_{QUANTUM_BAND.lower()}_density_xy": QUANTUM_DENSITY_XY_FIGURE,
            f"quantum_{QUANTUM_BAND.lower()}_density_x_line": QUANTUM_DENSITY_X_LINE_FIGURE,
            f"wavefunction_probability_state_{QUANTUM_STATE:04d}_xy": (
                QUANTUM_PROBABILITY_XY_FIGURE
            ),
            f"wavefunction_probability_state_{QUANTUM_STATE:04d}_x_line": (
                QUANTUM_PROBABILITY_X_LINE_FIGURE
            ),
            "quantum_occupation": QUANTUM_OCCUPATION_FIGURE,
            f"quantum_energy_spectrum_{QUANTUM_KPOINT}": QUANTUM_ENERGY_FIGURE,
        }
    )

display(pd.DataFrame({"selected_figure": list(SELECTED_FIGURES)}))

In [ ]:
FIGURE_OUTPUT_ROOT = Path(
    os.environ.get(
        "THESIS_FIGURE_ROOT",
        REPO_ROOT.parent
        / "qpu-local-outputs"
        / "refactoring"
        / "figures",
    )
).expanduser().resolve()

FIGURE_DIRECTORY = FIGURE_OUTPUT_ROOT / RUN_NAME / BIAS

FIGURE_FORMATS = ("pdf", "png")
FIGURE_PNG_DPI = 300

if FIGURE_OUTPUT_ROOT == REPO_ROOT or REPO_ROOT in FIGURE_OUTPUT_ROOT.parents:
    raise ValueError("FIGURE_OUTPUT_ROOT must be outside the repository.")

In [ ]:
if EXPORT_FIGURES:
    unsupported_figure_formats = sorted(set(FIGURE_FORMATS) - {"pdf", "png"})
    if unsupported_figure_formats:
        raise ValueError(
            "Unsupported figure format(s): "
            f"{unsupported_figure_formats}. Use only 'pdf' and 'png'."
        )

    for figure_stem, figure in SELECTED_FIGURES.items():
        if not isinstance(figure, Figure):
            raise TypeError(
                f"SELECTED_FIGURES[{figure_stem!r}] must be a Matplotlib Figure, "
                f"not {type(figure).__module__}.{type(figure).__name__}."
            )

    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    exported_figure_records = []
    exported_figure_files = {}

    for figure_stem, figure in SELECTED_FIGURES.items():
        figure_filenames = []
        for figure_format in FIGURE_FORMATS:
            figure_filename = f"{figure_stem}.{figure_format}"
            figure_path = FIGURE_DIRECTORY / figure_filename
            if figure_format == "png":
                figure.savefig(
                    figure_path,
                    bbox_inches="tight",
                    dpi=FIGURE_PNG_DPI,
                )
            else:
                figure.savefig(figure_path, bbox_inches="tight")

            figure_filenames.append(figure_filename)
            exported_figure_records.append(
                {"figure": figure_stem, "relative_path": figure_filename}
            )
        exported_figure_files[figure_stem] = figure_filenames

    FIGURE_MANIFEST = {
        "notebook_identifier": "simulation_results_browser",
        "run_directory": str(RUN_DIRECTORY),
        "run_name": RUN_NAME,
        "bias": BIAS,
        "xy_plane_z_nm": XY_PLANE_Z_NM,
        "xz_plane_y_nm": XZ_PLANE_Y_NM,
        "x_line_fixed_coordinates_nm": dict(X_LINE_FIXED_COORDINATES_NM),
        "z_line_fixed_coordinates_nm": dict(Z_LINE_FIXED_COORDINATES_NM),
        "quantum_analysis_enabled": ANALYSE_QUANTUM_OUTPUTS,
        "quantum": (
            {
                "region": QUANTUM_REGION,
                "band": QUANTUM_BAND,
                "kpoint": QUANTUM_KPOINT,
                "state": QUANTUM_STATE,
            }
            if ANALYSE_QUANTUM_OUTPUTS
            else None
        ),
        "selected_figure_stems": list(SELECTED_FIGURES),
        "figure_files": exported_figure_files,
        "formats": list(FIGURE_FORMATS),
        "png_dpi": FIGURE_PNG_DPI,
    }
    FIGURE_MANIFEST_PATH = FIGURE_DIRECTORY / "figure_manifest.json"
    FIGURE_MANIFEST_PATH.write_text(
        json.dumps(FIGURE_MANIFEST, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    EXPORTED_FIGURES = pd.DataFrame(exported_figure_records)
    display(EXPORTED_FIGURES)

## Summary

Review the selected dataset, cuts, quantum mode, and controlled export destination.

In [ ]:
NOTEBOOK_SUMMARY = pd.DataFrame(
    [
        {"setting": "run_directory", "value": str(RUN_DIRECTORY)},
        {"setting": "run_name", "value": RUN_NAME},
        {"setting": "bias", "value": BIAS},
        {"setting": "xy_plane_z_nm", "value": XY_PLANE_Z_NM},
        {"setting": "xz_plane_y_nm", "value": XZ_PLANE_Y_NM},
        {
            "setting": "x_line_fixed_coordinates_nm",
            "value": json.dumps(X_LINE_FIXED_COORDINATES_NM, sort_keys=True),
        },
        {
            "setting": "z_line_fixed_coordinates_nm",
            "value": json.dumps(Z_LINE_FIXED_COORDINATES_NM, sort_keys=True),
        },
        {
            "setting": "quantum_analysis_enabled",
            "value": ANALYSE_QUANTUM_OUTPUTS,
        },
        {"setting": "selected_figure_count", "value": len(SELECTED_FIGURES)},
        {"setting": "export_directory", "value": str(FIGURE_DIRECTORY)},
        {"setting": "export_enabled", "value": EXPORT_FIGURES},
    ]
)
display(NOTEBOOK_SUMMARY)